# Feature Engineering

This notebook converts raw candidate profiles into numerical features that can be used for candidate ranking.

In [70]:
import json
import pandas as pd
import numpy as np

In [71]:
NUM_CANDIDATES = 5000

candidates = []

with open("../data/candidates.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        candidates.append(json.loads(line))

        if len(candidates) == NUM_CANDIDATES:
            break

print(f"Loaded {len(candidates)} candidates.")

Loaded 5000 candidates.


# Feature Definitions

Each function extracts one measurable aspect of candidate suitability.

In [72]:
#function to calculate experience score
def experience_score(candidate):
    years = candidate["profile"]["years_of_experience"]

    if 5 <= years <= 9:
        return 1.0
    elif 4 <= years <= 12:
        return 0.7
    else:
        return 0.3

In [73]:
def open_to_work_score(candidate):
    return float(
        candidate["redrob_signals"]["open_to_work_flag"]
    )

In [74]:
def response_rate_score(candidate):
    return candidate["redrob_signals"][
        "recruiter_response_rate"
    ]

In [75]:
def notice_score(candidate):
    days = candidate["redrob_signals"][
        "notice_period_days"
    ]

    if days <= 30:
        return 1.0
    elif days <= 60:
        return 0.5
    else:
        return 0.0

In [76]:
def github_score(candidate):
    score = candidate["redrob_signals"][
        "github_activity_score"
    ]

    if score == -1:
        return 0

    return score / 100

In [77]:
def relocation_score(candidate):
    return float(
        candidate["redrob_signals"][
            "willing_to_relocate"
        ]
    )

In [78]:
EVIDENCE_WORDS = [
    "shipped",
    "production",
    "retrieval",
    "ranking",
    "recommendation",
    "search",
    "embedding",
    "evaluation",
    "a/b",
    "learning-to-rank",
    "offline metrics",
    "online metrics",
    "relevance"
]

In [79]:
def career_evidence_score(candidate):
    text = ""

    for job in candidate["career_history"]:
        text += (
            (job.get("title") or "")
            + " "
            + (job.get("description") or "")
            + " "
        )

    text = text.lower()

    score = 0

    for word in EVIDENCE_WORDS:
        if word in text:
            score += 1

    return score / len(EVIDENCE_WORDS)

In [80]:
KEYWORD_WEIGHTS = {
    "retrieval": 5,
    "ranking": 5,
    "recommendation": 5,
    "recommender": 5,
    "search": 4,
    "embedding": 4,
    "embeddings": 4,
    "vector": 3,
    "faiss": 5,
    "pinecone": 5,
    "milvus": 5,
    "qdrant": 5,
    "weaviate": 5,
    "elasticsearch": 4,
    "opensearch": 4,
    "semantic": 3
}

In [81]:
#Keyword score function
def keyword_relevance_score(candidate):
    text = ""

    profile = candidate["profile"]

    text += (profile.get("headline") or "") + " "
    text += (profile.get("summary") or "") + " "

    for job in candidate["career_history"]:
        text += (job.get("title") or "") + " "
        text += (job.get("description") or "") + " "

    for skill in candidate["skills"]:
        text += (skill.get("name") or "") + " "

    text = text.lower()

    score = 0

    for word, weight in KEYWORD_WEIGHTS.items():
        if word in text:
            score += weight

    return score

## Feature Extraction
Converting everything into one clear DataFrame

In [94]:
#building the dataframe
features = []

for c in candidates:
    features.append({
        "candidate_id": c["candidate_id"],
        "experience_score":
            experience_score(c),
        "keyword_relevance_score":
            keyword_relevance_score(c),
        "career_evidence_score":
            career_evidence_score(c),
        "open_to_work_score":
            open_to_work_score(c),
        "response_rate_score":
            response_rate_score(c),
        "notice_score":
            notice_score(c),
        "github_score":
            github_score(c),
        "relocation_score":
            relocation_score(c)
    })

features_df = pd.DataFrame(features)

features_df.head()

,candidate_id,experience_score,keyword_relevance_score,career_evidence_score,open_to_work_score,response_rate_score,notice_score,github_score,relocation_score
0,CAND_0000001,1.0,5,0.000000,1.0,0.34,0.5,0.092,0.0
1,CAND_0000002,0.3,4,0.153846,1.0,0.29,0.5,0.000,0.0
2,CAND_0000003,0.3,0,0.000000,0.0,0.46,0.0,0.000,0.0
3,CAND_0000004,0.3,4,0.153846,0.0,0.26,0.0,0.000,1.0
4,CAND_0000005,0.7,0,0.000000,1.0,0.37,1.0,0.000,1.0


In [143]:
print(features_df.describe())
print("\n")
print("Dataframe shape:", features_df.shape)

       experience_score  keyword_relevance_score  career_evidence_score  \
count       5000.000000              5000.000000            5000.000000   
mean           0.644060                 3.323200               0.079415   
std            0.301733                 6.936625               0.071382   
min            0.300000                 0.000000               0.000000   
25%            0.300000                 0.000000               0.000000   
50%            0.700000                 0.000000               0.076923   
75%            1.000000                 4.000000               0.153846   
max            1.000000                49.000000               0.692308   

       open_to_work_score  response_rate_score  notice_score  github_score  \
count          5000.00000          5000.000000   5000.000000   5000.000000   
mean              0.36720             0.436274      0.255800      0.101628   
std               0.48209             0.214272      0.359989      0.174438   
min         

In [141]:
print("Missing values in each column:")
print(features_df.isnull().sum())

Missing values in each column:
candidate_id               0
experience_score           0
keyword_relevance_score    0
career_evidence_score      0
open_to_work_score         0
response_rate_score        0
notice_score               0
github_score               0
relocation_score           0
keyword_score_norm         0
behavior_score             0
final_score                0
career_alignment_score     0
dtype: int64


## Keyword Normalization

Keyword scores have a much larger numerical range than the other handcrafted features.

Normalizing the keyword score ensures that no single feature dominates the final ranking and allows all signals to contribute proportionally.

In [148]:
features_df["keyword_relevance_score_norm"] = (
    features_df["keyword_relevance_score"]/ features_df["keyword_relevance_score"].max())

features_df["behavior_score"] = (
    0.4 * features_df["open_to_work_score"]
    + 0.4 * features_df["response_rate_score"]
    + 0.2 * features_df["notice_score"]
)

features_df["final_score"] = (
    0.30 * features_df["career_evidence_score"]
    + 0.25 * features_df["keyword_relevance_score_norm"]
    + 0.20 * features_df["behavior_score"]
    + 0.15 * features_df["experience_score"]
    + 0.05 * features_df["github_score"]
    + 0.05 * features_df["relocation_score"]
)

## Baseline Candidate Ranking

Combine handcrafted features into a weighted baseline ranking model.

In [144]:
ranked = features_df.sort_values(
    by="final_score",
    ascending=False
).reset_index(drop=True)

ranked.head(20)

,candidate_id,experience_score,keyword_relevance_score,career_evidence_score,open_to_work_score,response_rate_score,notice_score,github_score,relocation_score,keyword_score_norm,behavior_score,final_score,career_alignment_score
0,CAND_0002025,1.0,49,0.461538,1.0,0.80,1.0,0.969,0.0,1.000000,0.920,0.801835,1.0
1,CAND_0000031,1.0,37,0.615385,1.0,0.91,0.5,0.326,1.0,0.755102,0.864,0.800767,1.0
2,CAND_0001610,0.3,49,0.692308,1.0,0.57,0.0,0.400,1.0,1.000000,0.628,0.767277,1.0
3,CAND_0002120,1.0,33,0.230769,1.0,0.72,0.5,0.504,0.0,0.673469,0.788,0.635786,1.0
4,CAND_0004402,1.0,28,0.384615,1.0,0.83,0.5,0.000,0.0,0.571429,0.832,0.635240,1.0
5,CAND_0001494,1.0,27,0.230769,1.0,0.30,0.0,0.534,1.0,0.551020,0.520,0.622596,1.0
6,CAND_0001405,0.3,36,0.230769,1.0,0.69,0.0,0.633,1.0,0.734694,0.676,0.617681,1.0
7,CAND_0000666,1.0,24,0.384615,1.0,0.29,0.5,0.493,0.0,0.489796,0.616,0.611163,1.0
8,CAND_0001930,1.0,28,0.230769,1.0,0.65,0.0,0.763,0.0,0.571429,0.660,0.609128,1.0
9,CAND_0000273,1.0,20,0.230769,1.0,0.47,0.5,0.292,1.0,0.408163,0.688,0.607125,1.0


In [98]:
candidate_lookup = {
    c["candidate_id"]: c
    for c in candidates
}

top20 = ranked.head(20)

for cid in top20["candidate_id"]:
    c = candidate_lookup[cid]

    print("=" * 85)
    print("ID:", cid)
    print("Title:", c["profile"]["current_title"])
    print("Company:", c["profile"]["current_company"])
    print("Experience:", c["profile"]["years_of_experience"])
    print("Summary:")
    print(c["profile"]["summary"][:300])
    print()

ID: CAND_0002025
Title: Senior AI Engineer
Company: Apple
Experience: 5.9
Summary:
Senior AI engineer with 5.9 years of hands-on experience building production ML systems, with a focus on search, retrieval, and ranking. Most recently, I designed the company's first hybrid retrieval system combining BM25 with dense vector recall, across a corpus of 30M+ candidate profiles. My day-t

ID: CAND_0000031
Title: Recommendation Systems Engineer
Company: Swiggy
Experience: 6.0
Summary:
Machine learning engineer with 6.0 years of experience building ML-powered features in production. Strong background in NLP, recommendation systems, and applied AI; comfortable across the ML stack from feature engineering through deployment. Recently, I led the team that migrated our keyword-search-

ID: CAND_0001610
Title: Machine Learning Engineer
Company: Dream11
Experience: 3.0
Summary:
Machine learning engineer with 5.2 years of experience building ML-powered features in production. Strong background in NLP,

In [99]:
ranked.head(20)

,candidate_id,experience_score,keyword_relevance_score,career_evidence_score,open_to_work_score,response_rate_score,notice_score,github_score,relocation_score,keyword_relevance_score_norm,behavior_score,final_score
2024,CAND_0002025,1.0,49,0.461538,1.0,0.80,1.0,0.969,0.0,1.000000,0.920,0.770912
30,CAND_0000031,1.0,37,0.615385,1.0,0.91,0.5,0.326,1.0,0.755102,0.864,0.762491
1609,CAND_0001610,0.3,49,0.692308,1.0,0.57,0.0,0.400,1.0,1.000000,0.628,0.698292
4401,CAND_0004402,1.0,28,0.384615,1.0,0.83,0.5,0.000,0.0,0.571429,0.832,0.574642
2119,CAND_0002120,1.0,33,0.230769,1.0,0.72,0.5,0.504,0.0,0.673469,0.788,0.570398
1301,CAND_0001302,1.0,23,0.307692,1.0,0.80,0.5,0.305,0.0,0.469388,0.820,0.538905
1493,CAND_0001494,1.0,27,0.230769,1.0,0.30,0.0,0.534,1.0,0.551020,0.520,0.537686
665,CAND_0000666,1.0,24,0.384615,1.0,0.29,0.5,0.493,0.0,0.489796,0.616,0.535684
1929,CAND_0001930,1.0,28,0.230769,1.0,0.65,0.0,0.763,0.0,0.571429,0.660,0.532238
3056,CAND_0003057,1.0,30,0.153846,1.0,0.63,0.0,0.000,1.0,0.612245,0.652,0.529615


In [100]:
#inspection of the top 20 candidates
inspection_rows = []

candidate_lookup = {
    c["candidate_id"]: c
    for c in candidates
}

top20 = ranked.head(20)

for _, row in top20.iterrows():
    c = candidate_lookup[row["candidate_id"]]

    inspection_rows.append({
        "candidate_id": row["candidate_id"],
        "title": c["profile"]["current_title"],
        "company": c["profile"]["current_company"],
        "experience": c["profile"]["years_of_experience"],
        "final_score": round(row["final_score"], 3),
        "career_score": round(row["career_evidence_score"], 3),
        "keyword_relevance_score": row["keyword_relevance_score"],
        "open_to_work": c["redrob_signals"]["open_to_work_flag"]
    })

inspection_df = pd.DataFrame(inspection_rows)

inspection_df

,candidate_id,title,company,experience,final_score,career_score,keyword_relevance_score,open_to_work
0,CAND_0002025,Senior AI Engineer,Apple,5.9,0.771,0.462,49,True
1,CAND_0000031,Recommendation Systems Engineer,Swiggy,6.0,0.762,0.615,37,True
2,CAND_0001610,Machine Learning Engineer,Dream11,3.0,0.698,0.692,49,True
3,CAND_0004402,AI Research Engineer,Yellow.ai,6.0,0.575,0.385,28,True
4,CAND_0002120,ML Engineer,HCL,6.5,0.570,0.231,33,True
5,CAND_0001302,Computer Vision Engineer,Paytm,5.8,0.539,0.308,23,True
6,CAND_0001494,Data Scientist,Haptik,6.4,0.538,0.231,27,True
7,CAND_0000666,Data Scientist,PhonePe,6.5,0.536,0.385,24,True
8,CAND_0001930,Senior Software Engineer (ML),Dream11,6.9,0.532,0.231,28,True
9,CAND_0003057,Accountant,Infosys,7.8,0.530,0.154,30,True


## Career Alignment

Although career evidence improves ranking, some non-AI profiles with AI-related keywords are still appearing near the top.

To overcome this, a career alignment feature is introduced to reward AI-related job titles while penalizing unrelated roles such as Sales, Marketing, HR and Accounting.

In [102]:
def career_alignment_score(candidate):
    title = candidate["profile"]["current_title"].lower()

    positive_keywords = [
        "ai",
        "machine learning",
        "ml",
        "recommendation",
        "search",
        "nlp",
        "data scientist",
        "applied",
        "retrieval",
        "backend"
    ]

    negative_keywords = [
        "accountant",
        "sales",
        "marketing",
        "graphic",
        "content",
        "hr",
        "operations",
        "project manager",
        "business analyst",
        "customer support",
        "mechanical",
        "civil"
    ]

    if any(word in title for word in negative_keywords):
        return 0.0

    if any(word in title for word in positive_keywords):
        return 1.0

    return 0.5

In [103]:
features_df["career_alignment_score"] = [
    career_alignment_score(c)
    for c in candidates
]

In [150]:
features_df["final_score"] = (
    0.25 * features_df["career_evidence_score"]
    + 0.20 * features_df["keyword_relevance_score_norm"]
    + 0.20 * features_df["career_alignment_score"]
    + 0.15 * features_df["behavior_score"]
    + 0.10 * features_df["experience_score"]
    + 0.05 * features_df["github_score"]
    + 0.05 * features_df["relocation_score"]
)

ranked = (
    features_df
    .sort_values("final_score", ascending=False)
    .reset_index(drop=True)
)

ranked.head(20)

,candidate_id,experience_score,keyword_relevance_score,career_evidence_score,open_to_work_score,response_rate_score,notice_score,github_score,relocation_score,keyword_score_norm,behavior_score,final_score,career_alignment_score,keyword_relevance_score_norm
0,CAND_0002025,1.0,49,0.461538,1.0,0.80,1.0,0.969,0.0,1.000000,0.920,0.801835,1.0,1.000000
1,CAND_0000031,1.0,37,0.615385,1.0,0.91,0.5,0.326,1.0,0.755102,0.864,0.800767,1.0,0.755102
2,CAND_0001610,0.3,49,0.692308,1.0,0.57,0.0,0.400,1.0,1.000000,0.628,0.767277,1.0,1.000000
3,CAND_0002120,1.0,33,0.230769,1.0,0.72,0.5,0.504,0.0,0.673469,0.788,0.635786,1.0,0.673469
4,CAND_0004402,1.0,28,0.384615,1.0,0.83,0.5,0.000,0.0,0.571429,0.832,0.635240,1.0,0.571429
5,CAND_0001494,1.0,27,0.230769,1.0,0.30,0.0,0.534,1.0,0.551020,0.520,0.622596,1.0,0.551020
6,CAND_0001405,0.3,36,0.230769,1.0,0.69,0.0,0.633,1.0,0.734694,0.676,0.617681,1.0,0.734694
7,CAND_0000666,1.0,24,0.384615,1.0,0.29,0.5,0.493,0.0,0.489796,0.616,0.611163,1.0,0.489796
8,CAND_0001930,1.0,28,0.230769,1.0,0.65,0.0,0.763,0.0,0.571429,0.660,0.609128,1.0,0.571429
9,CAND_0000273,1.0,20,0.230769,1.0,0.47,0.5,0.292,1.0,0.408163,0.688,0.607125,1.0,0.408163


# Key Takeaways

- Career history provides stronger evidence than isolated AI keywords.
- Behavioral signals improve recruiter-oriented ranking.
- Career alignment reduces false positives from unrelated professions.
- The handcrafted feature pipeline serves as the baseline ranking model for the final candidate ranking system.